In [ ]:
# Fase 3 baseline: duck harness (Tufa Labs) en la G4 — validación offline CORTA.
import json, os, pickle, subprocess, sys, sysconfig, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
# CUDA linker path para vLLM/torch en imagen Kaggle
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
# Corte de validación offline (minutos) para NO gastar 9h de G4 cuando no es rerun.
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "120"))
WORKING = Path("/kaggle/working"); WORKING.mkdir(parents=True, exist_ok=True)
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


In [ ]:
# Instalar arc-agi del wheelhouse de la competencia (offline)
COMP_ROOT = None
for dp, dn, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dn:
        COMP_ROOT = Path(dp); break
assert COMP_ROOT, "wheelhouse no encontrado"
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi; print("arc_agi OK")

# Localizar el bundle del solver por su marker
BUNDLE = None
for m in Path("/kaggle/input").rglob("taaf-kaggle-bundle.json"):
    BUNDLE = m.parent; break
assert BUNDLE, "bundle TAAF no encontrado (adjunta thtennant/taaf-kaggle-source-share-fork)"
print("BUNDLE =", BUNDLE)

# Mapear datasets adjuntos a sus mounts
DATASET_SOURCES = ["thtennant/taaf-kaggle-source-share-fork",
                   "driessmit1/arc3-vllm-h100-wheelhouse-v3",
                   "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
def mount(ref):
    o,s = ref.split("/",1)
    for c in (Path("/kaggle/input")/s, Path("/kaggle/input/datasets")/o/s):
        if c.exists(): return str(c)
    return str(Path("/kaggle/input")/s)
paths = {r: (str(BUNDLE) if i==0 else mount(r)) for i,r in enumerate(DATASET_SOURCES)}
env_extra = {"TAAF_KAGGLE_INPUT_PATHS": json.dumps(paths, sort_keys=True),
             "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
             "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([])}
os.environ.update(env_extra)
SETUP_ENV = WORKING/"taaf_setup_env.json"; SETUP_ENV.write_text(json.dumps(env_extra))
print(paths)


In [ ]:
# Importar repos del bundle y correr setup_commands (instala vLLM, arranca el server)
def source_entries(b):
    out=[]
    for repo in sorted((b/"src").iterdir(), reverse=True):
        for c in (repo/"src", repo):
            if c.is_dir(): out.append(c)
    return out
entries = source_entries(BUNDLE)
for e in entries: sys.path.insert(0, str(e))
pth = Path(sysconfig.get_paths()["purelib"])/"taaf_sources.pth"
pth.write_text("".join(f"{e}\n" for e in entries))

def cmd_env():
    env = os.environ.copy(); env["PYTHON"]=sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"]=str(BUNDLE); env["TAAF_KAGGLE_WORKING_DIR"]=str(WORKING)
    env["TAAF_KAGGLE_SETUP_ENV"]=str(SETUP_ENV)
    env.update({str(k):str(v) for k,v in json.loads(SETUP_ENV.read_text()).items()})
    return env
env = cmd_env()
for c in json.loads((BUNDLE/"setup_commands.json").read_text()):
    print("setup:", c[:80], flush=True)
    subprocess.run(c, shell=True, check=True, cwd=WORKING, env=env)
    env = cmd_env(); os.environ.update(env)
for e in reversed([x for x in os.environ.get("PYTHONPATH","").split(os.pathsep) if x]):
    if e not in sys.path: sys.path.insert(0, e)
print("setup completo")


In [ ]:
# Cargar benchmark + target, jugar (offline recortado / gateway en rerun)
with open(BUNDLE/"deploy_target.pkl","rb") as f: target = pickle.load(f)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION
with open(BUNDLE/"benchmark_initial.pkl","rb") as f: bm = pickle.load(f)
bm.job_dir = WORKING; bm.n_passes = 1; bm.game_weights = None
os.environ.setdefault("RECORDINGS_DIR", str(WORKING/"server_recording"))

# Graft install (base = config v12 de thtennant, que marco 1.17) + schema_helpers:
# precarga helpers de analisis testeados (grid_diff, connected_components,
# action_effect_summary, recent_history) en el sandbox python del agente — el 27B
# reescribe esa plomeria con bugs en cada juego. NUESTRA tesis de feature injection,
# implementada por el autor del fork como graft sin habilitar (WP3).
# goalkeep NO va: marco 0.81 en el set oculto (-0.36 vs v12; ver working notes
# 2026-08-12). Blindado: cualquier fallo -> stock.
# Verificado en CPU local con scripts/smoke_graft_install.py (banner + prelude 8KB).
try:
    from taaf_grafts.composite import install as _graft_install
    _graft_install(bm, flags={"efficiency": True, "retry_guard": True,
                              "shortcircuit": True, "schema_helpers": True,
                              "banking": True})
except Exception as exc:
    print(f"[taaf_grafts] graft failed, running stock: {type(exc).__name__}: {exc}")

# CARGA DEL SEAM C (v2 de la amplificacion). La v1 inyectaba una FUNCION en el
# sandbox: se adoptaba (726 llamadas en 25/25 juegos) pero costaba un turno
# llamarla y devolvia vacio en los juegos sin movimiento. Esta v2 entrega el DATO
# YA CALCULADO como texto en el prompt: cero turnos, cero llamadas al sandbox, y
# nota no vacia en 25/25 juegos locales.
#
# Evidencia (banco micro, T4, 2026-08-19): con la tabla de efectos medida la
# planificacion sube de 44.0% a 66.1% en Qwen3-4B y de 24 items discordantes los
# 24 van a favor de la tabla, ninguno en contra (p aprox 0). OJO: a 1.7B el mismo
# dato PERJUDICA (15 vs 5, p=0.041) — hay un umbral de capacidad, y el modelo de
# produccion (27B) esta por encima de ambos.
#
# El detector esta validado por prediccion fuera de muestra sobre los 25 juegos
# locales: 96.6% (141/146) con confianza >= 0.6. Por debajo de ese umbral la nota
# NO afirma un desplazamiento, degrada a incertidumbre honesta.
try:
    import base64 as _b64
    import taaf_grafts.schema_helpers as _sh
    _ns = {}
    exec(compile(_b64.b64decode("IiIiTW9kZWxvIGRlIGVmZWN0b3MgTUVESURPIGEgcGFydGlyIGRlbCBoaXN0b3JpYWw6IGxhIGNhcmdhIGRlbCBzZWFtIEMuCgpRVUUgUFJPQkxFTUEgUkVTVUVMVkUuIExhIHYxIGRlIGxhIGFtcGxpZmljYWNpb24gaW55ZWN0byB1bmEgKmZ1bmNpb24qCihgcGxhbl9tb3Zlc2ApIHBvciBlbCBwcmVsdWRlIGRlbCBzYW5kYm94LiBTZSBhZG9wdG8gKDcyNiBsbGFtYWRhcyBlbiAyNS8yNQpqdWVnb3MgZGVzZGUgdW5hIHNvbGEgbGluZWEgZGUgbm90YSksIHBlcm8gbGEgY2FyZ2EgZXN0YWJhIG1hbCBlbGVnaWRhIHBvciBkb3MKcmF6b25lczoKCiAgMS4gQ09TVEFCQSBVTiBUVVJOTy4gRWwgbW9kZWxvIHRlbmlhIHF1ZSBlc2NyaWJpciBjb2RpZ28geSBlamVjdXRhcmxvIHBhcmEKICAgICBlbnRlcmFyc2UgZGUgYWxnbyBxdWUgbm9zb3Ryb3MgeWEgcG9kaWFtb3MgY2FsY3VsYXIgcG9yIGVsLgogIDIuIEVSQSBWQUNJQSBGVUVSQSBERSBKVUVHT1MgREUgTU9WSU1JRU5UTy4gU2kgbmFkYSBzZSB0cmFzbGFkYSwgYHBsYW5fbW92ZXNgCiAgICAgbm8gZGV2b2x2aWEgbmFkYSB1dGlsIHkgZWwgdHVybm8gc2UgcGVyZGlhIGVudGVyby4KCkVzdGUgbW9kdWxvIGNvcnJpZ2UgbGFzIGRvcy4gQ2FsY3VsYSBsYSB0YWJsYSBkZSBlZmVjdG9zIHBvciBhY2Npb24gbGV5ZW5kbyBsb3MKYEhpc3RvcnlFbnRyeWAgcXVlIGVsIGFnZW50ZSBZQSB0aWVuZSAoYGFjdGlvbmAsIGBmcmFtZS5ncmlkYCksIGFzaSBxdWU6CgogIC0gY3Vlc3RhIENFUk8gdHVybm9zIHkgQ0VSTyBsbGFtYWRhcyBhbCBzYW5kYm94OiBlbCBkYXRvIGxsZWdhIHlhIG1hc3RpY2FkbwogICAgZW4gZWwgcHJvbXB0IChzZWFtIEMsIGBfYnVpbGRfdXNlcl9wcm9tcHRgKTsKICAtIE5VTkNBIGVzdGEgdmFjaWE6IHNpIHVuYSBhY2Npb24gbm8gdHJhc2xhZGEgbmFkYSwgaW5mb3JtYSBpZ3VhbG1lbnRlIHNpCiAgICBjYW1iaWEgZWwgdGFibGVybyAoImNhbWJpYSIpIG8gc2kgbm8gaGFjZSBuYWRhICgic2luIGVmZWN0byIpLiBTYWJlciBxdWUKICAgIEFDVElPTjUgZXMgaW5lcnRlIGVuIGVzdGUganVlZ28gZXMgaW5mb3JtYWNpb24gYWNjaW9uYWJsZSDigJQgZGVqYSBkZSBnYXN0YXJzZQogICAgcHJlc3VwdWVzdG8gZW4gZWxsYS4KCkxhIGRldGVjY2lvbiBkZSB0cmFzbGFjaW9uIGV4Y2x1eWUgZWwgY29sb3IgZGUgZm9uZG8gKG1heW9yaXRhcmlvKS4gU2luIGVzYQpleGNsdXNpb24gZWwgZGV0ZWN0b3Igc2lndWUgYWwgZm9uZG8geSBhcHJlbmRlIGxvcyBzaWdub3MgaW52ZXJ0aWRvczogZXNlIGJ1ZwpyZWFsIGNvc3RvIHVuYSB0YXJkZSB5IGxvIGNhem8gc2NyaXB0cy90ZXN0X3NhbmRib3hfbmF2LnB5IGFudGVzIGRlIGdhc3RhciBHUFUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpNQVhfU0hJRlQgPSA4ICAgICAgICAgICMgcmFkaW8gZGUgYnVzcXVlZGEgZGUgbGEgdHJhc2xhY2lvbgpNSU5fQ0VMTFMgPSAxICAgICAgICAgICMgMSBiYXN0YTogZWwganVnYWRvciBkZSBVTkEgY2VsZGEgZXMgZWwgY2FzbyBtYXMgY29tdW4KICAgICAgICAgICAgICAgICAgICAgICAjIChtZWRpZG86IHR1OTMgeSBmdDA5IGNhbWJpYW4gZXhhY3RhbWVudGUgMiBjZWxkYXMgcG9yIHBhc28KICAgICAgICAgICAgICAgICAgICAgICAjID0gdW5hIHNlIHZhY2lhIHkgb3RyYSBzZSBsbGVuYSkuIEV4aWdpciAyIGxvcyBkZXNjYXJ0YWJhIGEKICAgICAgICAgICAgICAgICAgICAgICAjIHRvZG9zLiBFbCBlbXBhcmVqYW1pZW50byBwb3IgY29sb3IgeWEgZmlsdHJhIGVsIHJ1aWRvOiBzaQogICAgICAgICAgICAgICAgICAgICAgICMgbG9zIGNvbG9yZXMgbm8gY2FzYW4sIG5vIGhheSBjYW5kaWRhdG8geSBkZXZ1ZWx2ZSBOb25lLgpNQVhfRElGRl9DRUxMUyA9IDQwMCAgICMgbWFzIGNhbWJpbyBxdWUgZXN0byA9IHJlcGludGFkby9uaXZlbCBudWV2bywgbm8gdHJhc2xhY2lvbgpNSU5fQ09ORiA9IDAuNiAgICAgICAgICMgdW1icmFsIE1FRElETyBwb3IgcHJlZGljY2lvbiBmdWVyYSBkZSBtdWVzdHJhIHNvYnJlIGxvcyAyNQogICAgICAgICAgICAgICAgICAgICAgICMganVlZ29zIGxvY2FsZXM6IGNvbmY+PTAuNiAtPiA5Ni42JSAoMTQxLzE0Nik7IGNvbmY+PTAuNSAtPgogICAgICAgICAgICAgICAgICAgICAgICMgODguMSU7IHNpbiBmaWx0cm8gLT4gODUuMSUuIEVsIHNhbHRvIGVudHJlIDAuNiB5IDAuNSBlcwogICAgICAgICAgICAgICAgICAgICAgICMgbGltcGlvLCBhc2kgcXVlIGFoaSBzZSBjb3J0YS4KU0lNUExFX0FDVElPTlMgPSAoIkFDVElPTjEiLCAiQUNUSU9OMiIsICJBQ1RJT04zIiwgIkFDVElPTjQiLCAiQUNUSU9ONSIpCgoKZGVmIF9yb3dzKGdyaWQ6IEFueSkgLT4gbGlzdFtsaXN0W2ludF1dOgogICAgaWYgZ3JpZCBpcyBOb25lOgogICAgICAgIHJldHVybiBbXQogICAgdHJ5OgogICAgICAgIHJldHVybiBbW2ludCh2KSBmb3IgdiBpbiByb3ddIGZvciByb3cgaW4gZ3JpZF0KICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICByZXR1cm4gW10KCgpkZWYgYmFja2dyb3VuZF9vZihncmlkOiBsaXN0W2xpc3RbaW50XV0pIC0+IGludDoKICAgIGZyZXE6IENvdW50ZXIgPSBDb3VudGVyKCkKICAgIGZvciByb3cgaW4gZ3JpZDoKICAgICAgICBmcmVxLnVwZGF0ZShyb3cpCiAgICByZXR1cm4gZnJlcS5tb3N0X2NvbW1vbigxKVswXVswXSBpZiBmcmVxIGVsc2UgMAoKCmRlZiBzaGlmdF9iZXR3ZWVuKGJlZm9yZTogQW55LCBhZnRlcjogQW55KSAtPiB0dXBsZVtpbnQsIGludF0gfCBOb25lOgogICAgIiIiVHJhc2xhY2lvbiAoZHIsIGRjKSBxdWUgZXhwbGljYSBlbCBjYW1iaW8sIG8gTm9uZSBzaSBubyBsYSBoYXkuCgogICAgU09MTyBNSVJBIExBUyBDRUxEQVMgUVVFIENBTUJJQVJPTi4gRXN0YSBlcyBsYSBjb3JyZWNjaW9uIHF1ZSBoaXpvIGZhbHRhOgogICAgbG9zIHRhYmxlcm9zIHJlYWxlcyBzb24gZGVuc29zIChtZWRpZG86IDYzMC04NTUgY2VsZGFzIG5vLWZvbmRvIGRlIDQwOTYpLCBhc2kKICAgIHF1ZSBidXNjYXIgZWwgZGVzcGxhemFtaWVudG8gcXVlIG1lam9yIGFsaW5lYSBUT0RPIGVsIGNvbmp1bnRvIG5vLWZvbmRvIGVzCiAgICBhanVzdGFyIHJ1aWRvIOKAlCBzb2JyZSB1bmEgdGV4dHVyYSBkZW5zYSBzaWVtcHJlIGhheSBhbGd1biBvZmZzZXQgcXVlIGFsaW5lYQogICAgbXVjaGFzIGNlbGRhcyBwb3IgY2FzdWFsaWRhZC4gRG9zIGRldGVjdG9yZXMgY29uc3RydWlkb3MgYXNpIHNlIGNvbnRyYWRlY2lhbgogICAgZW50cmUgc2kgZW4gY2FzaSB0b2RvcyBsb3MgcGFyZXMsIHkgdW5vIHJlcG9ydGFiYSBlbCBNSVNNTyBkZXNwbGF6YW1pZW50byBwYXJhCiAgICBjdWF0cm8gYWNjaW9uZXMgZGlzdGludGFzLgoKICAgIExvIHF1ZSBkZSB2ZXJkYWQgb2N1cnJlIGN1YW5kbyB1biBvYmpldG8gc2UgbXVldmU6IHVuYXMgcG9jYXMgY2VsZGFzIHNlIHZhY2lhbgogICAgKGh1ZWxsYSB2aWVqYSkgeSBvdHJhcyBwb2NhcyBzZSBsbGVuYW4gKGh1ZWxsYSBudWV2YSk7IGxhcyA2MDArIHJlc3RhbnRlcyBubwogICAgY2FtYmlhbi4gQWxpbmVhbmRvIGVzYXMgZG9zIGh1ZWxsYXMgZWwgcHJvYmxlbWEgcXVlZGEgZGV0ZXJtaW5hZG8uCgogICAgTG9zIGNhbmRpZGF0b3Mgc2UgZ2VuZXJhbiBkZSBsb3MgcHJvcGlvcyBwYXJlcyBkZSBpZ3VhbCBjb2xvciAobm8gZGUgdW4gYmFycmlkbwogICAgZGUgY2FqYSksIGFzaSBxdWUgZGV0ZWN0YSBzYWx0b3MgZ3JhbmRlcyBzaW4gY29zdGUgZXh0cmEuCiAgICAiIiIKICAgIGIsIGEgPSBfcm93cyhiZWZvcmUpLCBfcm93cyhhZnRlcikKICAgIGlmIG5vdCBiIG9yIG5vdCBhIG9yIGxlbihiKSAhPSBsZW4oYSkgb3IgbGVuKGJbMF0pICE9IGxlbihhWzBdKToKICAgICAgICByZXR1cm4gTm9uZQogICAgY2hhbmdlZDogbGlzdFt0dXBsZVtpbnQsIGludF1dID0gW10KICAgIGZvciByIGluIHJhbmdlKGxlbihiKSk6CiAgICAgICAgcm93X2IsIHJvd19hID0gYltyXSwgYVtyXQogICAgICAgIGZvciBjIGluIHJhbmdlKG1pbihsZW4ocm93X2IpLCBsZW4ocm93X2EpKSk6CiAgICAgICAgICAgIGlmIHJvd19iW2NdICE9IHJvd19hW2NdOgogICAgICAgICAgICAgICAgY2hhbmdlZC5hcHBlbmQoKHIsIGMpKQogICAgaWYgbm90IGNoYW5nZWQgb3IgbGVuKGNoYW5nZWQpID4gTUFYX0RJRkZfQ0VMTFM6CiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICAjIENvbnRlbyBnbG9iYWw6IGVsIG9iamV0byBxdWUgc2UgbXVldmUgZXMgbG8gUkFSTzsgZWwgY2FtcG8gZXMgbG8gYWJ1bmRhbnRlLgogICAgIyBTaXJ2ZSBkZSBkZXNlbXBhdGUgY3VhbmRvIGRvcyBsZWN0dXJhcyBzb24gZ2VvbWV0cmljYW1lbnRlIHZhbGlkYXMgKHF1ZSBsYQogICAgIyBiYXJyYSBkZSBjb2xvciA5IGF2YW56bywgbyBxdWUgZWwgaHVlY28gZGUgY29sb3IgMCByZXRyb2NlZGlvKS4KICAgIHRvdGFsOiBDb3VudGVyID0gQ291bnRlcigpCiAgICBmb3Igcm93IGluIGI6CiAgICAgICAgdG90YWwudXBkYXRlKHJvdykKCiAgICBiZXN0X3NoaWZ0LCBiZXN0X2hpdHMsIGJlc3RfcmFyaXR5ID0gTm9uZSwgMCwgTm9uZQogICAgZm9yIGNvbG9yIGluIHtiW3JdW2NdIGZvciByLCBjIGluIGNoYW5nZWR9OgogICAgICAgIHNyYyA9IFsociwgYykgZm9yIHIsIGMgaW4gY2hhbmdlZCBpZiBiW3JdW2NdID09IGNvbG9yXQogICAgICAgIGRzdCA9IHsociwgYykgZm9yIHIsIGMgaW4gY2hhbmdlZCBpZiBhW3JdW2NdID09IGNvbG9yfQogICAgICAgIGlmIGxlbihzcmMpIDwgTUlOX0NFTExTIG9yIGxlbihkc3QpIDwgTUlOX0NFTExTOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNhbmQ6IENvdW50ZXIgPSBDb3VudGVyKCkKICAgICAgICBmb3Igc3IsIHNjIGluIHNyYzoKICAgICAgICAgICAgZm9yIGRyXywgZGNfIGluIGRzdDoKICAgICAgICAgICAgICAgIGQgPSAoZHJfIC0gc3IsIGRjXyAtIHNjKQogICAgICAgICAgICAgICAgaWYgZCAhPSAoMCwgMCkgYW5kIGFicyhkWzBdKSA8PSBNQVhfU0hJRlQgYW5kIGFicyhkWzFdKSA8PSBNQVhfU0hJRlQ6CiAgICAgICAgICAgICAgICAgICAgY2FuZFtkXSArPSAxCiAgICAgICAgaWYgbm90IGNhbmQ6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgc2hpZnQsIF8gPSBjYW5kLm1vc3RfY29tbW9uKDEpWzBdCiAgICAgICAgaGl0cyA9IHN1bSgxIGZvciByLCBjIGluIHNyYyBpZiAociArIHNoaWZ0WzBdLCBjICsgc2hpZnRbMV0pIGluIGRzdCkKICAgICAgICAjIGxhIHRyYXNsYWNpb24gZGViZSBleHBsaWNhciBsYSBtYXlvcmlhIGRlIGxhIGh1ZWxsYSBtYXMgcGVxdWVuYQogICAgICAgIGlmIGhpdHMgKiAyIDwgbWluKGxlbihzcmMpLCBsZW4oZHN0KSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcmFyaXR5ID0gdG90YWwuZ2V0KGNvbG9yLCAwKQogICAgICAgIGlmIGJlc3Rfc2hpZnQgaXMgTm9uZSBvciAocmFyaXR5LCAtaGl0cykgPCAoYmVzdF9yYXJpdHksIC1iZXN0X2hpdHMpOgogICAgICAgICAgICBiZXN0X3NoaWZ0LCBiZXN0X2hpdHMsIGJlc3RfcmFyaXR5ID0gc2hpZnQsIGhpdHMsIHJhcml0eQogICAgcmV0dXJuIGJlc3Rfc2hpZnQKCgpkZWYgZWZmZWN0c19mcm9tX2hpc3RvcnkoZW50cmllczogbGlzdFtBbnldLCB3aW5kb3c6IGludCA9IDQwKSAtPiBkaWN0W3N0ciwgZGljdF06CiAgICAiIiJBZ3JlZ2EgZWwgZWZlY3RvIG9ic2VydmFkbyBkZSBjYWRhIGFjY2lvbiBzb2JyZSBwYXJlcyBjb25zZWN1dGl2b3MuCgogICAgYGVudHJpZXNgIHNvbiBIaXN0b3J5RW50cnkgKGBhY3Rpb246IHN0cmAsIGBmcmFtZS5ncmlkYCkuIERldnVlbHZlIHBvciBhY2Npb246CiAgICAgIHsia2luZCI6ICJtb3ZlInwiY2FtYmlhInwic2luIGVmZWN0byIsICJzaGlmdCI6IFtkcixkY118Tm9uZSwgIm4iOiB2ZWNlcywgImNvbmYiOiAwLi4xfQoKICAgIFZFTlRBTkEgREUgUkVDRU5DSUEgKGB3aW5kb3dgKTogc29sbyBsYXMgdWx0aW1hcyB0cmFuc2ljaW9uZXMuIExvcyBlZmVjdG9zIHNvbgogICAgZGVwZW5kaWVudGVzIGRlbCBlc3RhZG8g4oCUIG1lZGlkbzogdmFyaW9zIGp1ZWdvcyBhcnJhbmNhbiBlbiB1bmEgcGFudGFsbGEgZG9uZGUKICAgIE5JTkdVTkEgYWNjaW9uIHNpbXBsZSBoYWNlIG5hZGEsIHkgc29sbyBkZXNwdWVzIHJlc3BvbmRlbi4gU2luIHZlbnRhbmEsIGVzZQogICAgInRvZG8gaW5lcnRlIiBkZWwgYXJyYW5xdWUgZW52ZW5lbmFyaWEgZWwgY29uc2VqbyBkdXJhbnRlIGVsIHJlc3RvIGRlIGxhCiAgICBwYXJ0aWRhLiBDb24gdmVudGFuYSwgbGEgdGFibGEgc2lndWUgYWwganVlZ28uCiAgICAiIiIKICAgIGlmIHdpbmRvdyBhbmQgbGVuKGVudHJpZXMgb3IgW10pID4gd2luZG93ICsgMToKICAgICAgICBlbnRyaWVzID0gZW50cmllc1stKHdpbmRvdyArIDEpOl0KICAgIG9iczogZGljdFtzdHIsIGxpc3RbdHVwbGVbaW50LCBpbnRdIHwgTm9uZV1dID0ge30KICAgIGNoYW5nZWQ6IGRpY3Rbc3RyLCBsaXN0W2Jvb2xdXSA9IHt9CiAgICBwcmV2ID0gTm9uZQogICAgZm9yIGUgaW4gZW50cmllcyBvciBbXToKICAgICAgICBncmlkID0gZ2V0YXR0cihnZXRhdHRyKGUsICJmcmFtZSIsIE5vbmUpLCAiZ3JpZCIsIE5vbmUpCiAgICAgICAgYWN0aW9uID0gZ2V0YXR0cihlLCAiYWN0aW9uIiwgTm9uZSkKICAgICAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgcHJldiBpcyBub3QgTm9uZSBhbmQgYWN0aW9uOgogICAgICAgICAgICAjIExvcyBjbGljcyBsbGVnYW4gY29tbyAiTU9VU0Uocm93PVIsIGNvbD1DKSI6IGNhZGEgdW5vIHVuaWNvLCBhc2kgcXVlCiAgICAgICAgICAgICMgc2luIGNhbm9uaWNhbGl6YXIgY2FkYSBjbGljIHNlcmlhIHVuYSAiYWNjaW9uIiBjb24gbj0xIHkgZWwgZmlsdHJvCiAgICAgICAgICAgICMgbWluX29icyBsb3MgZGVzY2FydGFyaWEgdG9kb3Mg4oCUIGVsIGhpc3RvcmlhbCBkZSBjbGljcyBxdWVkYXJpYSBtdWRvLgogICAgICAgICAgICAjIFNlIGFncmVnYW4gYmFqbyBBQ1RJT042LiAoU29uZGVhZG8gZW4gbG9zIDI1IGp1ZWdvcyBsb2NhbGVzOiBsYQogICAgICAgICAgICAjIHJlc3B1ZXN0YSBhIGNsaWNzIGVzIHRvZG8tby1uYWRhIHBvciBlc3RhZG8sIG51bmNhIHBvciBjb2xvciwgYXNpCiAgICAgICAgICAgICMgcXVlIGVsIGFncmVnYWRvIGJpbmFyaW8gZXMgZWwgcmVzdW1lbiBjb3JyZWN0byDigJQgdW4gZGVzZ2xvc2UgcG9yCiAgICAgICAgICAgICMgY29sb3Igbm8gZGlzY3JpbWluYXJpYSBuYWRhLikKICAgICAgICAgICAgaWYgYWN0aW9uLnN0YXJ0c3dpdGgoIk1PVVNFKCIpOgogICAgICAgICAgICAgICAgYWN0aW9uID0gIkFDVElPTjYiCiAgICAgICAgICAgIGIsIGEgPSBfcm93cyhwcmV2KSwgX3Jvd3MoZ3JpZCkKICAgICAgICAgICAgaWYgYiBhbmQgYToKICAgICAgICAgICAgICAgIGNoYW5nZWQuc2V0ZGVmYXVsdChhY3Rpb24sIFtdKS5hcHBlbmQoYiAhPSBhKQogICAgICAgICAgICAgICAgb2JzLnNldGRlZmF1bHQoYWN0aW9uLCBbXSkuYXBwZW5kKHNoaWZ0X2JldHdlZW4ocHJldiwgZ3JpZCkpCiAgICAgICAgcHJldiA9IGdyaWQKCiAgICB0YWJsZTogZGljdFtzdHIsIGRpY3RdID0ge30KICAgIGZvciBhY3Rpb24sIHNoaWZ0cyBpbiBvYnMuaXRlbXMoKToKICAgICAgICBuID0gbGVuKHNoaWZ0cykKICAgICAgICByZWFsID0gW3MgZm9yIHMgaW4gc2hpZnRzIGlmIHMgaXMgbm90IE5vbmVdCiAgICAgICAgY2ggPSBjaGFuZ2VkLmdldChhY3Rpb24sIFtdKQogICAgICAgIGlmIHJlYWw6CiAgICAgICAgICAgIHRvcCwgY250ID0gQ291bnRlcihyZWFsKS5tb3N0X2NvbW1vbigxKVswXQogICAgICAgICAgICBpZiBjbnQgKiAyID49IGxlbihyZWFsKToKICAgICAgICAgICAgICAgIHRhYmxlW2FjdGlvbl0gPSB7ImtpbmQiOiAibW92ZSIsICJzaGlmdCI6IFt0b3BbMF0sIHRvcFsxXV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJuIjogbiwgImNvbmYiOiByb3VuZChjbnQgLyBuLCAyKX0KICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgY2ggYW5kIG5vdCBhbnkoY2gpOgogICAgICAgICAgICB0YWJsZVthY3Rpb25dID0geyJraW5kIjogInNpbiBlZmVjdG8iLCAic2hpZnQiOiBOb25lLCAibiI6IG4sICJjb25mIjogMS4wfQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGZyYWMgPSAoc3VtKGNoKSAvIGxlbihjaCkpIGlmIGNoIGVsc2UgMC4wCiAgICAgICAgICAgIHRhYmxlW2FjdGlvbl0gPSB7ImtpbmQiOiAiY2FtYmlhIiwgInNoaWZ0IjogTm9uZSwgIm4iOiBuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjb25mIjogcm91bmQoZnJhYywgMil9CiAgICByZXR1cm4gdGFibGUKCgpkZWYgZGlyX3dvcmRzKHNyOiBpbnQsIHNjOiBpbnQpIC0+IHN0cjoKICAgICIiIigwLC0zKSAtPiAnMyBhIGxhIGl6cXVpZXJkYScuCgogICAgTk8gZXMgY29zbWV0aWNhLiBNZWRpZG8gZW4gZWwgYmFuY28gbWljcm8gY29uIFF3ZW4zLTRCIHNvYnJlIDEwOSBwcm9ibGVtYXMgZGUKICAgIHBsYW5pZmljYWNpb246IGNvbiBlbCB2ZWN0b3IgY3J1ZG8gKCJtb3ZlIDAgLTMiKSBhY2llcnRhIDY2LjElOyBjb24gbGEgbWlzbWEKICAgIGluZm9ybWFjaW9uIGVuIHBhbGFicmFzLCA4Ni4yJS4gUGFyZWFkbywgMjQgaXRlbXMgYSBmYXZvciBkZSBsYXMgcGFsYWJyYXMKICAgIGNvbnRyYSAyIChwIGFwcm94IDApLiBJbnRlcnByZXRhciBlbCB2ZWN0b3IgY29uc3VtZSByYXpvbmFtaWVudG8gcXVlIGVsIG1vZGVsbwogICAgbmVjZXNpdGEgcGFyYSBsYSB0YXJlYTsgbm9tYnJhciBsYSBkaXJlY2Npb24gc2UgbG8gZGV2dWVsdmUuIEVsIGZvcm1hdG8gZ2FuYQogICAgYWRlbWFzIGVuIGxvcyBET1MgdGFtYW5vcyBwcm9iYWRvcyAoMS43QiB5IDRCKSwgYXNpIHF1ZSBsYSBkaXJlY2Npb24gZXMKICAgIGVzdHJ1Y3R1cmFsIGRlbCBwcm9tcHQgeSBubyB1biByYXNnbyBkZSB1biBtb2RlbG8gY29uY3JldG8uCiAgICAiIiIKICAgIHBhcnRlcyA9IFtdCiAgICBpZiBzciA8IDA6CiAgICAgICAgcGFydGVzLmFwcGVuZChmInstc3J9IGFycmliYSIpCiAgICBpZiBzciA+IDA6CiAgICAgICAgcGFydGVzLmFwcGVuZChmIntzcn0gYWJham8iKQogICAgaWYgc2MgPCAwOgogICAgICAgIHBhcnRlcy5hcHBlbmQoZiJ7LXNjfSBhIGxhIGl6cXVpZXJkYSIpCiAgICBpZiBzYyA+IDA6CiAgICAgICAgcGFydGVzLmFwcGVuZChmIntzY30gYSBsYSBkZXJlY2hhIikKICAgIHJldHVybiAiIHkgIi5qb2luKHBhcnRlcykgaWYgcGFydGVzIGVsc2UgIm5hZGEiCgoKZGVmIHJlbmRlcl9lZmZlY3RzX25vdGUodGFibGU6IGRpY3Rbc3RyLCBkaWN0XSwgbWluX29iczogaW50ID0gMikgLT4gc3RyOgogICAgIiIiQ29udmllcnRlIGxhIHRhYmxhIGVuIGxhcyBsaW5lYXMgZGUgdGV4dG8gcXVlIHNlIGFuZXhhbiBhbCBwcm9tcHQuCgogICAgU29sbyBzZSByZXBvcnRhbiBhY2Npb25lcyBjb24gb2JzZXJ2YWNpb25lcyBzdWZpY2llbnRlczogYWZpcm1hciB1biBlZmVjdG8KICAgIGEgcGFydGlyIGRlIHVuYSBzb2xhIG11ZXN0cmEgZXMgY29tbyBkZWNpZGlyIHVuIGV4cGVyaW1lbnRvIGNvbiBuPTEsIHF1ZSBlcwogICAgZXhhY3RhbWVudGUgZWwgZXJyb3IgcXVlIG5vcyBjb3N0byBjdWF0cm8gbm9jaGVzLgogICAgIiIiCiAgICBpZiBub3QgdGFibGU6CiAgICAgICAgcmV0dXJuICIiCiAgICBsaW5lcyA9IFtdCiAgICBmb3IgYWN0aW9uIGluIHNvcnRlZCh0YWJsZSk6CiAgICAgICAgZCA9IHRhYmxlW2FjdGlvbl0KICAgICAgICBpZiBkWyJuIl0gPCBtaW5fb2JzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIGRbImtpbmQiXSA9PSAibW92ZSIgYW5kIGRbImNvbmYiXSA+PSBNSU5fQ09ORjoKICAgICAgICAgICAgZHIsIGRjID0gZFsic2hpZnQiXQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiIgIHthY3Rpb259OiBtdWV2ZSB7ZGlyX3dvcmRzKGRyLCBkYyl9IgogICAgICAgICAgICAgICAgICAgICAgICAgZiIgIFt7ZFsnbiddfSBvYnMsIHtkWydjb25mJ106LjAlfSBjb25zaXN0ZW50ZV0iKQogICAgICAgIGVsaWYgZFsia2luZCJdID09ICJtb3ZlIjoKICAgICAgICAgICAgIyBNZWRpZG86IHBvciBkZWJham8gZGUgTUlOX0NPTkYgbGEgcHJlZGljY2lvbiBmdWVyYSBkZSBtdWVzdHJhIGNhZSBhCiAgICAgICAgICAgICMgfjYwJS4gQWZpcm1hciBhaGkgbWV0ZXJpYSBoZWNob3MgRkFMU09TIGVuIGVsIHByb21wdCwgcXVlIGVzIHBlb3IgcXVlCiAgICAgICAgICAgICMgY2FsbGFyLiBTZSBkZWdyYWRhIGEgaW5jZXJ0aWR1bWJyZSBob25lc3RhLgogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiIgIHthY3Rpb259OiBjYW1iaWEgZWwgdGFibGVybywgcGVybyBzdSBlZmVjdG8gTk8gZXMgIgogICAgICAgICAgICAgICAgICAgICAgICAgZiJjb25zdGFudGUgKHtkWydjb25mJ106LjAlfSBkZSB7ZFsnbiddfSBvYnMpIOKAlCB2ZXJpZmljYSBhbnRlcyBkZSBmaWFydGUiKQogICAgICAgIGVsaWYgZFsia2luZCJdID09ICJzaW4gZWZlY3RvIiBhbmQgYWN0aW9uID09ICJBQ1RJT042IjoKICAgICAgICAgICAgIyBBQ1RJT042IGVzIFBPU0lDSU9OQUw6IGZhbGxhciBlbiBOIGNlbGRhcyBubyBnZW5lcmFsaXphIGFsIGNhbmFsLgogICAgICAgICAgICAjIExhIHY3IHJlbmRlcml6YWJhIGFxdWkgZWwgbWlzbW8gZGVzY2FydGUgcXVlIHBhcmEgQUNUSU9OMS01IHkgcG9kaWEKICAgICAgICAgICAgIyBzdXByaW1pciBlbCB1bmljbyBjYW5hbCBkZSBjb250cm9sIGRlIHVuIGp1ZWdvIGRlIGNsaWNzIGNvbiBzb2xvIDIKICAgICAgICAgICAgIyBjbGljcyBkZXNhZm9ydHVuYWRvcyAobWluX29icz0yKS4gSGlwb3Rlc2lzIG1lY2FuaWNhIHJlZ2lzdHJhZGEgZW4KICAgICAgICAgICAgIyBERVNJR04gOC4xNiBBTlRFUyBkZSBsYSAyYSBtdWVzdHJhIGRlIHY3OyBjb3JyZWdpZG8gYSBjb25zZWpvCiAgICAgICAgICAgICMgcG9zaWNpb25hbCBxdWUgbnVuY2EgZGVzY2FydGEgZWwgY2FuYWwuCiAgICAgICAgICAgIGxpbmVzLmFwcGVuZChmIiAgQUNUSU9ONjogbG9zIGNsaWNzIHByb2JhZG9zICh7ZFsnbiddfSkgbm8gY2FtYmlhcm9uIG5hZGEsICIKICAgICAgICAgICAgICAgICAgICAgICAgIGYicGVybyB1biBjbGljIGRlcGVuZGUgZGUgbGEgQ0VMREEg4oCUIHBydWViYSBjZWxkYXMgZGlzdGludGFzICIKICAgICAgICAgICAgICAgICAgICAgICAgIGYiYW50ZXMgZGUgZGVzY2FydGFybG8iKQogICAgICAgIGVsaWYgZFsia2luZCJdID09ICJzaW4gZWZlY3RvIjoKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiICB7YWN0aW9ufTogU0lOIEVGRUNUTyBlbiB7ZFsnbiddfSBpbnRlbnRvcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICBmIuKAlCBubyBnYXN0ZXMgdHVybm9zIGVuIGVsbGEiKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGxpbmVzLmFwcGVuZChmIiAge2FjdGlvbn06IGNhbWJpYSBlbCB0YWJsZXJvIHNpbiB0cmFzbGFkYXIgbmFkYSAiCiAgICAgICAgICAgICAgICAgICAgICAgICBmIlt7ZFsnbiddfSBvYnNdIikKICAgIGlmIG5vdCBsaW5lczoKICAgICAgICByZXR1cm4gIiIKICAgIG5vdGUgPSAoIkVmZWN0byBNRURJRE8gZGUgY2FkYSBhY2Npb24gZW4gRVNUQSBwYXJ0aWRhIChjYWxjdWxhZG8gZGUgdHUgcHJvcGlvICIKICAgICAgICAgICAgImhpc3RvcmlhbCwgbm8gbG8gcmVjYWxjdWxlcyk6XG4iICsgIlxuIi5qb2luKGxpbmVzKSkKCiAgICAjIENhc28gbWVkaWRvIGVuIDUgZGUgMjUganVlZ29zIGxvY2FsZXM6IE5JTkdVTkEgYWNjaW9uIHNpbXBsZSBoYWNlIG5hZGEuIEFoaQogICAgIyBlbCBhZ2VudGUgcHVlZGUgcXVlbWFyIGxhIHBhcnRpZGEgZW50ZXJhIHB1bHNhbmRvIGJvdG9uZXMgbXVlcnRvcywgYXNpIHF1ZQogICAgIyBsYSBjb25jbHVzaW9uIHNlIGRpY2UgZXhwbGljaXRhbWVudGUgZW4gdmV6IGRlIGRlamFybGEgZGVkdWNpci4KICAgICMKICAgICMgUEVSTyBsYSByZWNvbWVuZGFjaW9uIGRlIEFDVElPTjYgZXMgY29uZGljaW9uYWwgYSBsYSBldmlkZW5jaWEgKHNvbmRlYWRvIGVuCiAgICAjIGxvcyAyNSBqdWVnb3MpOiBlbiBzNWk1L3ZjMzMvdG4zNiBsb3MgY2xpY3MgY2FtYmlhbiBlbCB0YWJsZXJvIHNpZW1wcmU7IGVuCiAgICAjIGxwODUvc3UxNSBOSSBsb3MgY2xpY3MgcHJvYmFkb3MgaGFjZW4gbmFkYSBlbiBlc2UgZXN0YWRvLiBSZWNvbWVuZGFyIGNsaWNzCiAgICAjIHNpbiBldmlkZW5jaWEgc2VyaWEgYWZpcm1hciB1biBoZWNobyBubyB2ZXJpZmljYWRvIOKAlCB5IGxhIHZhcmlhbnRlIEYgbWlkaW8KICAgICMgcXVlIGxhIGluY2VydGlkdW1icmUgaG9uZXN0YSBwcm90ZWdlICg0MSB2cyA1IGRpc2NvcmRhbnRlcyBhIGZhdm9yKS4KICAgIHNpbXBsZXMgPSBbYSBmb3IgYSBpbiB0YWJsZSBpZiBhIGluIFNJTVBMRV9BQ1RJT05TXQogICAgaWYgc2ltcGxlcyBhbmQgYWxsKHRhYmxlW2FdWyJraW5kIl0gPT0gInNpbiBlZmVjdG8iIGZvciBhIGluIHNpbXBsZXMpOgogICAgICAgIG5vdGUgKz0gIlxuICA9PiBOSU5HVU5BIGFjY2lvbiBzaW1wbGUgaGFjZSBuYWRhIGFxdWkuIE5vIGdhc3RlcyB0dXJub3MgZW4gQUNUSU9OMS01LiIKICAgICAgICBjbGlja3MgPSB0YWJsZS5nZXQoIkFDVElPTjYiKQogICAgICAgIGlmIGNsaWNrcyBhbmQgY2xpY2tzWyJuIl0gPj0gbWluX29icyBhbmQgY2xpY2tzWyJraW5kIl0gIT0gInNpbiBlZmVjdG8iOgogICAgICAgICAgICBub3RlICs9ICgiIExvcyBjbGljcyBTSSBjYW1iaWFuIGVsIHRhYmxlcm86IGVzdGUganVlZ28gc2UgY29udHJvbGEgIgogICAgICAgICAgICAgICAgICAgICAicG9yIGNvb3JkZW5hZGFzIChBQ1RJT042KS4iKQogICAgICAgIGVsaWYgY2xpY2tzIGFuZCBjbGlja3NbIm4iXSA+PSA4OgogICAgICAgICAgICAjIHNvbG8gY29uIE1VQ0hPUyBjbGljcyBmYWxsaWRvcyBzZSBzdWdpZXJlIGJ1c2NhciBvdHJhIGNvc2Eg4oCUIHkgYXVuCiAgICAgICAgICAgICMgYXNpIHNpbiBkZXNjYXJ0YXIgZWwgY2FuYWwsIHBvcnF1ZSBlbCBmYWxsbyBkZSBjbGljcyBlcyBwb3IgY2VsZGEKICAgICAgICAgICAgbm90ZSArPSAoZiIgTG9zIGNsaWNzIHByb2JhZG9zICh7Y2xpY2tzWyduJ119KSB0YW1wb2NvIGNhbWJpYXJvbiBuYWRhIGVuICIKICAgICAgICAgICAgICAgICAgICAgImVzdGUgZXN0YWRvOiBwcnVlYmEgY2VsZGFzIG11eSBkaXN0aW50YXMsIHkgc2kgbmFkYSByZXNwb25kZSwgIgogICAgICAgICAgICAgICAgICAgICAiY29uc2lkZXJhIGVzcGVyYXIgbyBSRVNFVC4iKQogICAgICAgIGVsaWYgY2xpY2tzIGFuZCBjbGlja3NbIm4iXSA+PSBtaW5fb2JzOgogICAgICAgICAgICBub3RlICs9IChmIiBMb3MgY2xpY3MgcHJvYmFkb3MgKHtjbGlja3NbJ24nXX0pIGF1biBubyBjYW1iaWFyb24gbmFkYSwgIgogICAgICAgICAgICAgICAgICAgICAicGVybyBzb24gcG9jb3MgeSBlbCBlZmVjdG8gZGVwZW5kZSBkZSBsYSBjZWxkYTogcHJ1ZWJhIGNlbGRhcyAiCiAgICAgICAgICAgICAgICAgICAgICJkaXN0aW50YXMgYW50ZXMgZGUgY29uY2x1aXIuIikKICAgICAgICBlbHNlOgogICAgICAgICAgICBub3RlICs9ICgiIExvcyBjbGljcyAoQUNUSU9ONikgYXVuIG5vIHNlIGhhbiBwcm9iYWRvIGxvIHN1ZmljaWVudGU6ICIKICAgICAgICAgICAgICAgICAgICAgInBydWViYWxvcywgcGVybyB2ZXJpZmljYSBxdWUgY2FtYmlhbiBhbGdvIGFudGVzIGRlIGluc2lzdGlyLiIpCiAgICByZXR1cm4gbm90ZQo=").decode("utf-8"), "effects_model.py", "exec"), _ns)
    _effects_from_history = _ns["effects_from_history"]
    _render_effects_note = _ns["render_effects_note"]
    _orig_bup = _sh.SchemaHelpersToolAgent._build_user_prompt

    def _bup_with_effects(self, action_num, **kw):
        base = _orig_bup(self, action_num, **kw)
        try:
            note = _render_effects_note(_effects_from_history(kw.get("history_entries") or []))
        except Exception:
            return base          # cualquier fallo => prompt del padre, intacto
        return f"{base}\n{note}" if note else base

    _sh.SchemaHelpersToolAgent._build_user_prompt = _bup_with_effects
    print("EFFECTS_NOTE injected on seam C:", len(_ns), "symbols")
except Exception as exc:
    print(f"[effects_note] injection failed, running stock: {type(exc).__name__}: {exc}")

import arc_agi, taaf.game_api
def games_offline(d):
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=d)
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]
def games_comp():
    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.COMPETITION,
                                    arc_base_url=os.environ["ARC_BASE_URL"], environments_dir="")
    ar = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.COMPETITION,
                        arc_base_url=spec.arc_base_url, environments_dir="")
    return [taaf.game_api.GameAPI(env_name=e.game_id, arcade_spec=spec) for e in ar.available_environments]

soft_end = None
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY","test-key-123")
    os.environ.setdefault("ARC_BASE_URL","http://gateway:8001/")
    dl = time.monotonic()+600
    while time.monotonic()<dl:
        try:
            with urlopen(os.environ["ARC_BASE_URL"]+"api/games", timeout=10) as r:
                if r.status<500: break
        except Exception: pass
        time.sleep(5)
    bm.games = games_comp()
else:
    bm.games = games_offline(str(COMP_ROOT/"environment_files"))
    soft_end = datetime.fromtimestamp(NOTEBOOK_START)+timedelta(minutes=OFFLINE_SOFT_MIN)

import pandas as pd
pd.DataFrame([["1_0","1",True,1]], columns=["row_id","game_id","end_of_game","score"]).to_parquet(WORKING/"submission.parquet", index=False)

try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
finally:
    for c in json.loads((BUNDLE/"teardown_commands.json").read_text()):
        subprocess.run(c, shell=True, check=False, cwd=WORKING, env=cmd_env())
print("run terminado")
